In [1]:
!pip install bm25s pymorphy3 pymorphy3-dicts-ru sentence-transformers datasets torch faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.7 MB/s eta 0:00:00


In [ ]:
import random
import numpy as np
import torch
import pandas as pd
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, util
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers
from sklearn.model_selection import train_test_split


TRAIN_PATH = "train.parquet" # тут путь до трейна
MODEL_NAME = "cointegrated/rubert-tiny2"
MAX_SEQ_LENGTH = 128
BATCH_SIZE = 32               
VAL_FRAC = 0.15
N_EPOCHS = 6 
TOP_K = 50
MODEL_OUTPUT_DIR = "rubert-tiny2-finetuned-full" # сюда сохранится модель, этот же путь нужен будет в следующем ноутбуке
random.seed(42)

def build_item_text(row) -> str: # конкатенируем название и описание итема
    parts = [row.get("item_title_raw"), row.get("item_description_raw")]
    return " ".join(p for p in parts if isinstance(p, str) and p)


def recall_at_k(predictions: dict, gt: dict, k: int = TOP_K) -> float: #непосредственно полнота
    scores = []
    for id, item in gt.items():
        if not item:
            continue
        pred = set(predictions.get(id, [])[:k])
        scores.append(len(pred & item) / len(item))
    return float(np.mean(scores)) if scores else 0.0


def evaluate(model: SentenceTransformer) -> float: #считаем
    item_emb = model.encode(item_texts, batch_size=256, convert_to_tensor=True,
                             normalize_embeddings=True, show_progress_bar=True).cpu()
    query_emb = model.encode(val_queries_list, batch_size=256, convert_to_tensor=True,
                              normalize_embeddings=True, show_progress_bar=False).cpu()
    hits = util.semantic_search(query_emb, item_emb, corpus_chunk_size=50_000, top_k=TOP_K)
    predictions = {q: [item_ids[h["corpus_id"]] for h in row]
                   for q, row in zip(val_queries_list, hits)}
    return recall_at_k(predictions, val_relevant)


# группируем запросы, потому что может быть много запросов типа "маникюр"
# и простой сплит может закинуть их и в трейн и в валидацию
train = pd.read_parquet(TRAIN_PATH)
train["query_text"] = train["search_query"].fillna("").str.strip().str.lower()
full_relevant_by_query = train.groupby("query_text")["item_id"].apply(set).to_dict()
query_texts_list = sorted(full_relevant_by_query.keys())
train_queries, val_queries = train_test_split(
    query_texts_list,
    test_size=VAL_FRAC,
    random_state=42
)
train_queries = set(train_queries)
val_queries = set(val_queries)
train_pairs = train[train["query_text"].isin(train_queries)].reset_index(drop=True)
val_relevant = {q: full_relevant_by_query[q] for q in val_queries}

all_items = train.drop_duplicates(subset="item_id").reset_index(drop=True)
item_text_by_id = dict(zip(all_items["item_id"], all_items.apply(build_item_text, axis=1)))
item_ids = all_items["item_id"].tolist()
item_texts = [item_text_by_id[i] for i in item_ids]
val_queries_list = sorted(val_relevant.keys())

model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

anchors = train_pairs["query_text"].tolist()
positives = [item_text_by_id[i] for i in train_pairs["item_id"]]
dataset = Dataset.from_dict({"anchor": anchors, "positive": positives})
best_recall = -1
for epoch in range(N_EPOCHS):
    args = SentenceTransformerTrainingArguments(
        output_dir = f"{MODEL_OUTPUT_DIR}/epoch_{epoch + 1}",
        num_train_epochs=1,
        per_device_train_batch_size=BATCH_SIZE,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        warmup_steps=0.1,
        learning_rate=2e-5,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        save_strategy="no",
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=loss)
    trainer.train()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    recall = evaluate(model)
    print(f"epoch {epoch}: recall@50 = {recall:.4f}")
    if recall > best_recall:
        best_recall = recall
        model.save(f"{MODEL_OUTPUT_DIR}/best")


print(f"Best recall@50 = {best_recall:.4f}")

/tmp/ipykernel_23/3746936775.py:7: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss
/tmp/ipykernel_23/3746936775.py:8: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

...

Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


epoch 4: recall@50 = 0.2632


Step,Training Loss
200,0.380976
400,0.279696
600,0.202452
800,0.143046
1000,0.143640
1200,0.142887
1400,0.148868
1600,0.152140
1800,0.158561
2000,0.173693


Batches:   0%|          | 0/1347 [00:00<?, ?it/s]

epoch 5: recall@50 = 0.2623
Best recall@50 = 0.2650
